In [2]:
import os
import random
from pathlib import Path
from time import sleep
from typing import cast

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from tqdm import tqdm

d:\Bluebird\Code 1\Agentic RAG\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [3]:
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "fiqa"
INDEX_DIR = ROOT_DIR / "indexes"
DENSE_DIR = INDEX_DIR / "dense"


In [5]:
def create_client():
    load_dotenv()
    os.environ["NVIDIA_API_KEY"] = cast(str, (os.getenv("NVIDIA_API_KEY")))
    client = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b", trucate="NONE")

    return client

In [6]:
text_to_embed = """This dude built 3,000 landing pages across 1,000 U.S. cities to generate leads through SEO.

He didn’t wait around for results either. Within a few weeks the pages were already ranking on their own"""

In [ ]:
client = create_client()
response = client.embed_documents([text_to_embed])
vector = np.asarray(response)
print("\n", len(response[0]), "\n", (vector.shape[1]))

In [6]:
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
docs_text = [t.strip() or '[Empty Document]' for t in corpus["text"].tolist()]
docs_id = [i.strip() for i in corpus["_id"].tolist()]
print(len(docs_text), len(docs_id))


57638 57638


In [7]:
CHECKPOINT_PATH=DENSE_DIR

In [9]:
def build_index(texts: list[str], batch_size: int = 256)->np.ndarray:
    chunks: list[np.ndarray] = []
    start_index: int = 0

    checkpoint_path=CHECKPOINT_PATH/"embeddings.npz"
    if checkpoint_path.exists():
        checkpoint=np.load(checkpoint_path)
        chunks=[checkpoint['embeddings']]
        start_index=int(checkpoint['next_index'])

    for i in tqdm(range(start_index, len(texts), batch_size),desc="Embedding"):
        batch_chunk = texts[i : i + batch_size]
        response=create_embed(batch_chunk)

        vector=np.asarray(response)
        chunks.append(vector)

        np.savez_compressed(checkpoint,embedding=np.vstack(chunks),next_index=start_index+len(batch_chunk))

    final_embedding=np.vstack(chunks)

    if len(texts)!=final_embedding.shape[1]:
        raise ValueError(f"Expected {len(texts)} recieved {final_embedding.shape[1]}")

    return final_embedding

      


In [4]:
final_emdbeddings=np.load(DENSE_DIR/"embeddings.npy")
final_emdbeddings.shape

(57638, 2048)

In [10]:
def create_embed(text: list[str], max_tries: int = 5) -> np.ndarray:
    client = create_client()
    for attempt in range(max_tries):
        try:
            response = client.embed_documents(text)
            vector = np.asarray(response)
            if len(response[0]) != vector.shape[0]:
                raise ValueError(
                    f"Expected: {len(response[0])} but recieved {vector.shape[0]}"
                )
            return vector
        except Exception as error:
            if attempt == max_tries - 1:
                delay = (2**attempt) + random.random()
                print(f"Error occured {error} retrying in {delay}")
                sleep(delay)
    raise ValueError("Something went wrong")

In [11]:
def dense_search(query_text:str,embed_doc:np.ndarray,k:int=10):
    embed_query=create_embed([query_text])
    query_vector=np.asarray(embed_query,dtype=np.float32)
    normalize_query=query_vector/np.linalg.norm(query_vector,keepdims=True)
    scores= embed_doc @ normalize_query 
    top_k=np.argsort(-scores)[:k]
    return [(docs_id[i],float(scores[i])) for i in top_k]




In [12]:
def main():
    query="Where should I park my rainy-day funds?"

    if (DENSE_DIR/"embeddings.npy").exists():
        final_embedding=np.load(DENSE_DIR/"embeddings.npy")
    else:
        DENSE_DIR.mkdir(exist_ok=True,parents=True)
        final_embedding=create_embed(docs_text)
        np.save(DENSE_DIR/"embeddings.npy",final_embedding)
        print("Embeddings saved")

    final_embedding_normalized=final_embedding/np.linalg.norm(final_embedding)

    for i,(doc_id,score)  in enumerate(dense_search(embed_doc=final_embedding_normalized,query_text=query,k=5),1):
        text=corpus.loc[corpus['_id']==doc_id,'text'].iloc[0]
        print(f"{i} [{score:3f} {text[:80]}]")



        



    

In [ ]:
if __name__=="__main__":
    main()

d:\Bluebird\Code 1\Agentic RAG\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(
